# MycoMetrics - Notebook 01: Cultivation & Yield EDA
## Biological Efficiency, Flush Drop-Off Dynamics, and Room Performance
**Business Focus**: FungiFlow Organics Cultivation Analysis  
**Author**: Data Analytics Team  

### Executive Overview
This notebook investigates biological efficiency distributions across cultivated mushroom varieties (Pearl Oyster, Lion's Mane, Shiitake), performs parametric normality checks, and runs ANOVA hypothesis testing to evaluate batch performance against scientific targets.

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to analytical warehouse
con = duckdb.connect('../data/warehouse.duckdb')
con.execute("SET search_path = 'analytics_analytics'")

batches = con.execute("""
    select 
        b.batch_id,
        m.mushroom_common_name,
        l.location_name as growing_room,
        b.cycle_duration_days,
        b.dry_substrate_kg,
        b.total_accepted_harvest_kg,
        b.biological_efficiency_pct,
        b.rejection_rate_pct,
        b.direct_material_cost_per_kg
    from fact_production_batches b
    inner join dim_mushroom m on b.mushroom_id = m.mushroom_id
    inner join dim_location l on b.location_id = l.location_id
""").df()

batches.head()

### 1. Descriptive Summary by Variety
We calculate mean, median, standard deviation, and interquartile range (IQR) for Biological Efficiency (%):

In [ ]:
desc_stats = batches.groupby('mushroom_common_name')['biological_efficiency_pct'].agg(
    count='count',
    mean='mean',
    std='std',
    median='median',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(2)
desc_stats

### 2. Normality Testing (Shapiro-Wilk)
Evaluating normality of Biological Efficiency:
$$H_0: \text{The data is normally distributed}$$
$$H_1: \text{The data deviates significantly from normality}$$

In [ ]:
for var in ['Pearl Oyster', "Lion's Mane", 'Shiitake']:
    sub = batches[batches['mushroom_common_name'] == var]['biological_efficiency_pct'].dropna()
    stat, p = stats.shapiro(sub)
    print(f"{var:15s} | Shapiro-Wilk W = {stat:.4f}, p-value = {p:.4f} -> {'Normal' if p > 0.05 else 'Non-Normal'}")

### 3. One-Way ANOVA: Variety Biological Efficiency Comparison
Testing whether mean Biological Efficiency differs significantly between varieties:
$$H_0: \mu_{Oyster} = \mu_{Lion's Mane} = \mu_{Shiitake}$$
$$H_1: \text{At least one variety mean differs}$$

In [ ]:
p_oyster = batches[batches['mushroom_common_name'] == 'Pearl Oyster']['biological_efficiency_pct'].dropna()
l_mane = batches[batches['mushroom_common_name'] == "Lion's Mane"]['biological_efficiency_pct'].dropna()
s_take = batches[batches['mushroom_common_name'] == 'Shiitake']['biological_efficiency_pct'].dropna()

f_stat, p_val = stats.f_oneway(p_oyster, l_mane, s_take)
kw_stat, kw_p = stats.kruskal(p_oyster, l_mane, s_take)

print(f"One-Way ANOVA:       F = {f_stat:.3f}, p = {p_val:.4e}")
print(f"Kruskal-Wallis Test: H = {kw_stat:.3f}, p = {kw_p:.4e}")